In [12]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src"

sys.path.append(str(SRC_DIR))

from load_data import load_data

In [13]:
X_train, X_test, y_train, y_test = load_data()
print(X_train.shape, X_test.shape)

# stratify가 제대로 되었는지: train/test 클래스 비율 비교
ratio = pd.DataFrame({
    "train": y_train.value_counts(normalize=True),
    "test": y_test.value_counts(normalize=True),
}).round(3)
print(ratio)

(1552, 26) (389, 26)
              train   test
target                    
Other_Faults  0.347  0.347
Bumps         0.207  0.208
K_Scratch     0.202  0.201
Z_Scratch     0.098  0.098
Pastry        0.081  0.082
Stains        0.037  0.036
Dirtiness     0.028  0.028


In [14]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)
class_names = le.classes_
print(dict(enumerate(class_names)))

{0: 'Bumps', 1: 'Dirtiness', 2: 'K_Scratch', 3: 'Other_Faults', 4: 'Pastry', 5: 'Stains', 6: 'Z_Scratch'}


In [15]:
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    "LogReg": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=2000, random_state=42)),
    ]),
    "DecisionTree": DecisionTreeClassifier(random_state=42),
    "RandomForest": RandomForestClassifier(random_state=42, n_jobs=-1),
    "GradientBoosting": GradientBoostingClassifier(random_state=42),
    "XGBoost": XGBClassifier(objective="multi:softprob", eval_metric="mlogloss",
                             random_state=42, n_jobs=-1),
}

In [16]:
from sklearn.model_selection import cross_validate

scoring = ["accuracy", "precision_macro", "recall_macro", "f1_macro"]

rows = []
for name, model in models.items():
    res = cross_validate(model, X_train, y_train_enc, cv=cv, scoring=scoring)
    row = {"model": name}
    for s in scoring:
        row[f"{s}_mean"] = res[f"test_{s}"].mean()
        row[f"{s}_std"] = res[f"test_{s}"].std()
    rows.append(row)

baseline_df = pd.DataFrame(rows).sort_values("f1_macro_mean", ascending=False)
baseline_df.round(3)

,model,accuracy_mean,accuracy_std,precision_macro_mean,precision_macro_std,recall_macro_mean,recall_macro_std,f1_macro_mean,f1_macro_std
4,XGBoost,0.794,0.017,0.835,0.021,0.800,0.040,0.811,0.031
3,GradientBoosting,0.778,0.008,0.814,0.009,0.781,0.025,0.794,0.015
2,RandomForest,0.764,0.014,0.813,0.019,0.758,0.031,0.779,0.025
0,LogReg,0.715,0.017,0.733,0.023,0.719,0.032,0.721,0.024
1,DecisionTree,0.711,0.024,0.714,0.030,0.732,0.032,0.720,0.029


In [19]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import classification_report, confusion_matrix

best_name = "XGBoost"   # Cell 5 결과를 보고 직접 지정
best_model = models[best_name]

y_pred_cv = cross_val_predict(best_model, X_train, y_train_enc, cv=cv)

print(classification_report(y_train_enc, y_pred_cv,
                            target_names=class_names, digits=3))

cm = pd.DataFrame(confusion_matrix(y_train_enc, y_pred_cv),
                  index=class_names, columns=class_names)
cm

              precision    recall  f1-score   support

       Bumps      0.670     0.682     0.676       321
   Dirtiness      0.921     0.795     0.854        44
   K_Scratch      0.971     0.965     0.968       313
Other_Faults      0.733     0.771     0.752       538
      Pastry      0.627     0.508     0.561       126
      Stains      0.947     0.931     0.939        58
   Z_Scratch      0.954     0.947     0.950       152

    accuracy                          0.794      1552
   macro avg      0.832     0.800     0.814      1552
weighted avg      0.794     0.794     0.794      1552



,Bumps,Dirtiness,K_Scratch,Other_Faults,Pastry,Stains,Z_Scratch
Bumps,219,2,0,87,11,1,1
Dirtiness,2,35,0,5,2,0,0
K_Scratch,1,0,302,8,1,1,0
Other_Faults,85,1,8,415,24,1,4
Pastry,18,0,0,42,64,0,2
Stains,1,0,0,3,0,54,0
Z_Scratch,1,0,1,6,0,0,144


In [20]:
from sklearn.inspection import permutation_importance
from sklearn.base import clone

imp_list = []
for tr_idx, va_idx in cv.split(X_train, y_train_enc):
    m = clone(best_model)
    m.fit(X_train.iloc[tr_idx], y_train_enc[tr_idx])
    r = permutation_importance(m, X_train.iloc[va_idx], y_train_enc[va_idx],
                               scoring="f1_macro", n_repeats=10,
                               random_state=42, n_jobs=-1)
    imp_list.append(r.importances_mean)

imp = pd.DataFrame(imp_list, columns=X_train.columns)
imp_summary = (pd.DataFrame({"mean": imp.mean(), "std": imp.std()})
               .sort_values("mean", ascending=False))
imp_summary.round(4)

,mean,std
Steel_Plate_Thickness,0.1296,0.0426
Length_of_Conveyer,0.0968,0.0307
Pixels_Areas,0.0941,0.0200
TypeOfSteel_A300,0.0731,0.0091
Y_Minimum,0.0570,0.0188
Orientation_Index,0.0492,0.0183
Minimum_of_Luminosity,0.0464,0.0142
Square_Index,0.0448,0.0169
X_Minimum,0.0265,0.0208
Edges_Y_Index,0.0241,0.0172


In [21]:
print(pd.crosstab(y_train, X_train["TypeOfSteel_A300"], normalize="columns").round(2))
print("Length_of_Conveyer 고유값 수:", X_train["Length_of_Conveyer"].nunique())
print(X_train.groupby(y_train)["Length_of_Conveyer"].describe().round(0))

TypeOfSteel_A300     0     1
target                      
Bumps             0.11  0.36
Dirtiness         0.04  0.01
K_Scratch         0.34  0.00
Other_Faults      0.35  0.34
Pastry            0.09  0.06
Stains            0.06  0.00
Z_Scratch         0.02  0.22
Length_of_Conveyer 고유값 수: 80
              count    mean    std     min     25%     50%     75%     max
target                                                                    
Bumps         321.0  1520.0  162.0  1227.0  1362.0  1624.0  1690.0  1794.0
Dirtiness      44.0  1428.0  127.0  1352.0  1358.0  1364.0  1366.0  1696.0
K_Scratch     313.0  1384.0   50.0  1356.0  1358.0  1362.0  1400.0  1670.0
Other_Faults  538.0  1483.0  153.0  1227.0  1360.0  1372.0  1687.0  1715.0
Pastry        126.0  1550.0  156.0  1333.0  1362.0  1647.0  1688.0  1710.0
Stains         58.0  1358.0    2.0  1354.0  1358.0  1358.0  1360.0  1362.0
Z_Scratch     152.0  1358.0   29.0  1348.0  1354.0  1356.0  1358.0  1708.0


In [22]:
from sklearn.metrics import roc_auc_score

def pair_auc(a, b):
    mask = y_train.isin([a, b])
    Xp, yp = X_train[mask], (y_train[mask] == a).astype(int)
    auc = {c: roc_auc_score(yp, Xp[c]) for c in Xp.columns}
    s = pd.Series(auc)
    return (s - 0.5).abs().add(0.5).sort_values(ascending=False)  # 방향 무관 구분력

pair_df = pd.DataFrame({
    "Bumps_vs_Other": pair_auc("Bumps", "Other_Faults"),
    "Pastry_vs_Other": pair_auc("Pastry", "Other_Faults"),
})
pair_df.round(3)

,Bumps_vs_Other,Pastry_vs_Other
Edges_Index,0.599,0.575
Edges_X_Index,0.570,0.656
Edges_Y_Index,0.641,0.766
Empty_Index,0.626,0.707
Length_of_Conveyer,0.559,0.625
LogOfAreas,0.567,0.592
Log_X_Index,0.569,0.714
Log_Y_Index,0.574,0.696
Luminosity_Index,0.562,0.650
Maximum_of_Luminosity,0.509,0.552


In [23]:
key = ["TypeOfSteel_A300", "Steel_Plate_Thickness", "Length_of_Conveyer"]

g_train = X_train.groupby(key).size()
print("Train 조합 수:", len(g_train))
print(g_train.describe().round(1))

# 이 조합 안에서 클래스가 얼마나 한쪽으로 몰리는지
tmp = X_train[key].copy()
tmp["target"] = y_train.values
purity = tmp.groupby(key)["target"].agg(lambda s: s.value_counts(normalize=True).iloc[0])
print("조합 내 최빈 클래스 비율 (가중 평균):",
      round((purity * g_train).sum() / g_train.sum(), 3))

# Test의 조합이 Train에도 있는지
test_keys = set(map(tuple, X_test[key].values))
train_keys = set(g_train.index)
print("Test 조합 중 Train에도 있는 비율:", round(len(test_keys & train_keys) / len(test_keys), 3))

Train 조합 수: 218
count    218.0
mean       7.1
std       12.5
min        1.0
25%        1.0
50%        3.0
75%        7.0
max      108.0
dtype: float64
조합 내 최빈 클래스 비율 (가중 평균): 0.763
Test 조합 중 Train에도 있는 비율: 0.844


In [24]:
def add_features(X):
    X = X.copy()
    mean_lum = X["Sum_of_Luminosity"] / X["Pixels_Areas"]
    X["Luminosity_Range"] = X["Maximum_of_Luminosity"] - X["Minimum_of_Luminosity"]
    X["Dark_Depth"] = mean_lum - X["Minimum_of_Luminosity"]
    X["Compactness"] = (X["X_Perimeter"] + X["Y_Perimeter"]) / np.sqrt(X["Pixels_Areas"])
    return X

X_train_fe = add_features(X_train)
new_cols = ["Luminosity_Range", "Dark_Depth", "Compactness"]

# 쌍별 구분력 (기존 pair_auc를 X_train_fe 기준으로)
def pair_auc_fe(a, b, cols):
    mask = y_train.isin([a, b])
    yp = (y_train[mask] == a).astype(int)
    s = pd.Series({c: roc_auc_score(yp, X_train_fe.loc[mask, c]) for c in cols})
    return (s - 0.5).abs().add(0.5)

print(pd.DataFrame({
    "Bumps_vs_Other": pair_auc_fe("Bumps", "Other_Faults", new_cols),
    "Pastry_vs_Other": pair_auc_fe("Pastry", "Other_Faults", new_cols),
}).round(3))

# 기존 변수와의 최대 상관 (중복 확인)
corr = X_train_fe.corr().abs()
for c in new_cols:
    print(c, "→", corr[c].drop(new_cols).idxmax(), round(corr[c].drop(new_cols).max(), 3))

                  Bumps_vs_Other  Pastry_vs_Other
Luminosity_Range           0.633            0.757
Dark_Depth                 0.631            0.760
Compactness                0.683            0.558
Luminosity_Range → Minimum_of_Luminosity 0.817
Dark_Depth → Minimum_of_Luminosity 0.802
Compactness → Y_Perimeter 0.824


In [25]:
from sklearn.base import clone

for label, Xtr in [("baseline", X_train), ("+FE", X_train_fe)]:
    m = clone(models["XGBoost"])
    res = cross_validate(m, Xtr, y_train_enc, cv=cv, scoring=["f1_macro"])
    pred = cross_val_predict(clone(models["XGBoost"]), Xtr, y_train_enc, cv=cv)
    cm = pd.DataFrame(confusion_matrix(y_train_enc, pred), index=class_names, columns=class_names)
    print(f"{label}: Macro F1 {res['test_f1_macro'].mean():.3f} ± {res['test_f1_macro'].std():.3f} | "
          f"Pastry→Other {cm.loc['Pastry','Other_Faults']} | "
          f"Bumps↔Other {cm.loc['Bumps','Other_Faults'] + cm.loc['Other_Faults','Bumps']}")

baseline: Macro F1 0.811 ± 0.031 | Pastry→Other 42 | Bumps↔Other 172
+FE: Macro F1 0.814 ± 0.020 | Pastry→Other 37 | Bumps↔Other 174


In [26]:
rng = np.random.default_rng(42)
group_sizes = tmp.groupby(key).size()

def weighted_purity(labels):
    t = tmp[key].copy()
    t["target"] = labels
    p = t.groupby(key)["target"].agg(lambda s: s.value_counts(normalize=True).iloc[0])
    return (p * group_sizes).sum() / group_sizes.sum()

perm = [weighted_purity(rng.permutation(y_train.values)) for _ in range(50)]
print("실제 순도:", round(weighted_purity(y_train.values), 3))
print("라벨 섞었을 때 순도: 평균", round(np.mean(perm), 3), "/ 최대", round(np.max(perm), 3))

실제 순도: 0.763
라벨 섞었을 때 순도: 평균 0.431 / 최대 0.446


In [27]:
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score

groups = X_train[key].astype(str).agg("_".join, axis=1)
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

rows = {}
for label, splitter, g in [("Stratified", cv, None), ("Group", sgkf, groups)]:
    res = cross_validate(clone(models["XGBoost"]), X_train, y_train_enc,
                         cv=splitter, groups=g, scoring="f1_macro")
    pred = cross_val_predict(clone(models["XGBoost"]), X_train, y_train_enc,
                             cv=splitter, groups=g)
    print(f"{label}: Macro F1 {res['test_score'].mean():.3f} ± {res['test_score'].std():.3f}")
    rows[label] = f1_score(y_train_enc, pred, average=None)

pd.DataFrame(rows, index=class_names).round(3)

Stratified: Macro F1 0.811 ± 0.031
Group: Macro F1 0.720 ± 0.044


,Stratified,Group
Bumps,0.676,0.635
Dirtiness,0.854,0.738
K_Scratch,0.968,0.903
Other_Faults,0.752,0.670
Pastry,0.561,0.543
Stains,0.939,0.947
Z_Scratch,0.950,0.844


In [28]:
X_train_fe = add_features(X_train)

for cv_name, splitter, g in [("Stratified", cv, None), ("Group", sgkf, groups)]:
    for label, Xtr in [("baseline", X_train), ("+FE", X_train_fe)]:
        res = cross_validate(clone(models["XGBoost"]), Xtr, y_train_enc,
                             cv=splitter, groups=g, scoring="f1_macro")
        pred = cross_val_predict(clone(models["XGBoost"]), Xtr, y_train_enc,
                                 cv=splitter, groups=g)
        cm = pd.DataFrame(confusion_matrix(y_train_enc, pred),
                          index=class_names, columns=class_names)
        print(f"[{cv_name}] {label}: Macro F1 {res['test_score'].mean():.3f} "
              f"± {res['test_score'].std():.3f} | "
              f"Pastry→Other {cm.loc['Pastry','Other_Faults']} | "
              f"Bumps↔Other {cm.loc['Bumps','Other_Faults'] + cm.loc['Other_Faults','Bumps']}")

[Stratified] baseline: Macro F1 0.811 ± 0.031 | Pastry→Other 42 | Bumps↔Other 172
[Stratified] +FE: Macro F1 0.814 ± 0.020 | Pastry→Other 37 | Bumps↔Other 174
[Group] baseline: Macro F1 0.720 ± 0.044 | Pastry→Other 45 | Bumps↔Other 196
[Group] +FE: Macro F1 0.732 ± 0.032 | Pastry→Other 41 | Bumps↔Other 196


In [29]:
def add_features_v2(X):
    X = add_features(X)   # 기존 3개
    X["Perimeter_Ratio"] = X["X_Perimeter"] / X["Y_Perimeter"].clip(lower=1)
    X["Log_X_Perimeter"] = np.log10(X["X_Perimeter"].clip(lower=1))
    X["Log_Y_Perimeter"] = np.log10(X["Y_Perimeter"].clip(lower=1))
    X["Rel_X_Center"] = (X["X_Minimum"] + X["X_Maximum"]) / 2 / X["Length_of_Conveyer"]
    lum_sum = (X["Maximum_of_Luminosity"] + X["Minimum_of_Luminosity"]).clip(lower=1)
    X["Lum_Contrast"] = (X["Maximum_of_Luminosity"] - X["Minimum_of_Luminosity"]) / lum_sum
    return X

feature_sets = {
    "baseline": X_train,
    "+FE3": add_features(X_train),
    "+FE8": add_features_v2(X_train),
}

records = []
for model_name in ["LogReg", "XGBoost"]:
    for cv_name, splitter, g in [("Stratified", cv, None), ("Group", sgkf, groups)]:
        for set_name, Xtr in feature_sets.items():
            res = cross_validate(clone(models[model_name]), Xtr, y_train_enc,
                                 cv=splitter, groups=g, scoring="f1_macro")
            records.append({
                "model": model_name, "cv": cv_name, "features": set_name,
                "f1": f"{res['test_score'].mean():.3f} ± {res['test_score'].std():.3f}",
            })

pd.DataFrame(records).pivot(index=["model", "cv"], columns="features", values="f1")

features                     +FE3           +FE8       baseline
model   cv                                                     
LogReg  Group       0.605 ± 0.072  0.612 ± 0.074  0.609 ± 0.062
        Stratified  0.723 ± 0.024  0.731 ± 0.030  0.721 ± 0.024
XGBoost Group       0.732 ± 0.032  0.741 ± 0.031  0.720 ± 0.044
        Stratified  0.814 ± 0.020  0.817 ± 0.013  0.811 ± 0.031

In [30]:
X_fe8 = add_features_v2(X_train)
diffs = {"Stratified": [], "Group": []}

for seed in [0, 1, 2]:
    splitters = {
        "Stratified": (StratifiedKFold(5, shuffle=True, random_state=seed), None),
        "Group": (StratifiedGroupKFold(5, shuffle=True, random_state=seed), groups),
    }
    for name, (sp, g) in splitters.items():
        b = cross_validate(clone(models["XGBoost"]), X_train, y_train_enc,
                           cv=sp, groups=g, scoring="f1_macro")["test_score"]
        f = cross_validate(clone(models["XGBoost"]), X_fe8, y_train_enc,
                           cv=sp, groups=g, scoring="f1_macro")["test_score"]
        diffs[name].extend(f - b)

for name, d in diffs.items():
    d = np.array(d)
    print(f"{name}: 평균 차이 {d.mean():+.3f} | FE8이 더 높은 fold {(d > 0).sum()} / {len(d)}")

Stratified: 평균 차이 +0.004 | FE8이 더 높은 fold 10 / 15
Group: 평균 차이 +0.017 | FE8이 더 높은 fold 11 / 15


In [31]:
def add_position(X):
    X = X.copy()
    L = X["Length_of_Conveyer"]
    x_center = (X["X_Minimum"] + X["X_Maximum"]) / 2
    X["x_rel"] = x_center / L
    X["edge_dist_px"] = np.minimum(X["X_Minimum"], L - X["X_Maximum"])
    X["left_side"] = (X["x_rel"] < 0.5).astype(int)
    X["edge_band10"] = ((X["x_rel"] < 0.1) | (X["x_rel"] > 0.9)).astype(int)
    return X

X_pos = add_position(X_train)

for model_name in ["LogReg", "XGBoost"]:
    diffs = {"Stratified": [], "Group": []}
    for seed in [0, 1, 2]:
        splitters = {
            "Stratified": (StratifiedKFold(5, shuffle=True, random_state=seed), None),
            "Group": (StratifiedGroupKFold(5, shuffle=True, random_state=seed), groups),
        }
        for name, (sp, g) in splitters.items():
            b = cross_validate(clone(models[model_name]), X_train, y_train_enc,
                               cv=sp, groups=g, scoring="f1_macro")["test_score"]
            f = cross_validate(clone(models[model_name]), X_pos, y_train_enc,
                               cv=sp, groups=g, scoring="f1_macro")["test_score"]
            diffs[name].extend(f - b)
    for name, d in diffs.items():
        d = np.array(d)
        print(f"{model_name} [{name}] 평균 차이 {d.mean():+.3f} | 이긴 fold {(d > 0).sum()}/15")

LogReg [Stratified] 평균 차이 +0.008 | 이긴 fold 11/15
LogReg [Group] 평균 차이 +0.015 | 이긴 fold 10/15
XGBoost [Stratified] 평균 차이 -0.000 | 이긴 fold 7/15
XGBoost [Group] 평균 차이 +0.000 | 이긴 fold 8/15
